In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('Размер train:', train.shape)
print('Размер test:', test.shape)

train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isnull().sum()

In [ ]:
print(train['price_range'].value_counts())

In [ ]:
train['price_range'].value_counts().plot(kind='bar')

plt.title('Количество телефонов в каждом ценовом классе')
plt.xlabel('Класс цены')
plt.ylabel('Количество')

plt.show()

In [ ]:
corr = train.corr()

plt.figure(figsize=(16, 10))

plt.imshow(corr, cmap='coolwarm', aspect='auto')
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title('Матрица корреляции')

plt.show()

In [ ]:
price_corr = corr['price_range'].sort_values(ascending=False)

print(price_corr)

In [ ]:
price_corr.drop('price_range').plot(kind='bar', figsize=(12,6))

plt.title('Корреляция признаков с price_range')
plt.xlabel('Признаки')
plt.ylabel('Корреляция')

plt.grid()

plt.show()

In [ ]:
for i in range(4):
    subset = train[train['price_range'] == i]
    
    plt.hist(subset['ram'], alpha=0.5, bins=20, label=f'Class {i}')

plt.legend()

plt.title('Распределение RAM по классам')
plt.xlabel('RAM')
plt.ylabel('Количество')

plt.show()

In [ ]:
X = train.drop('price_range', axis=1)
y = train['price_range']

In [ ]:
#train test split омагад 
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

X_test_scaled = scaler.transform(test[X.columns])

In [ ]:
lr_model = LogisticRegression()

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_valid_scaled)
print(lr_pred)

In [ ]:
print("Accuracy:", accuracy_score(y_valid, lr_pred))

In [ ]:
print(classification_report(y_valid, lr_pred))

In [ ]:
print(confusion_matrix(y_valid, lr_pred))

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

knn_pred = knn_model.predict(X_valid_scaled)

print("KNN Accuracy:", accuracy_score(y_valid, knn_pred))

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_valid)

print("Decision Tree Accuracy:",
      accuracy_score(y_valid, tree_pred))

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_valid)

print("Random Forest Accuracy:",
      accuracy_score(y_valid, rf_pred))

In [ ]:
gb_model = GradientBoostingClassifier(
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_valid)

print("Gradient Boosting Accuracy:",
      accuracy_score(y_valid, gb_pred))

In [ ]:
svm_model = SVC(kernel='rbf')

svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_valid_scaled)

print("SVM Accuracy:",
      accuracy_score(y_valid, svm_pred))

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'KNN',
        'Decision Tree',
        'Random Forest',
        'Gradient Boosting',
        'SVM'
    ],

    'Accuracy': [
        accuracy_score(y_valid, lr_pred),
        accuracy_score(y_valid, knn_pred),
        accuracy_score(y_valid, tree_pred),
        accuracy_score(y_valid, rf_pred),
        accuracy_score(y_valid, gb_pred),
        accuracy_score(y_valid, svm_pred)
    ]
})

print(results)

In [ ]:
results.sort_values('Accuracy').plot(
    x='Model',
    y='Accuracy',
    kind='barh',
    figsize=(10,6)
)

plt.title('Сравнение моделей')

plt.xlabel('Accuracy')

plt.ylabel('Модель')

plt.grid()

plt.show()

In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    'Importance',
    ascending=False
)

print(importance)

In [ ]:
importance.plot(
    x='Feature',
    y='Importance',
    kind='bar',
    figsize=(12,6)
)

plt.title('Важность признаков')

plt.ylabel('Importance')

plt.grid()

plt.show()

In [ ]:
rf_probs = rf_model.predict_proba(X_valid)
print(rf_probs)

In [ ]:
valid_df = X_valid.copy()

valid_df['real_class'] = y_valid.values

valid_df['predicted_class'] = rf_pred

In [ ]:
valid_df['max_probability'] = np.max(rf_probs, axis=1)

In [ ]:
valid_df['second_class'] = np.argsort(rf_probs, axis=1)[:, -2]

In [ ]:
sorted_probs = np.sort(rf_probs, axis=1)

valid_df['prob_difference'] = (
    sorted_probs[:, -1] - sorted_probs[:, -2]
)

In [ ]:
anomalies = valid_df[
    (valid_df['real_class'] != valid_df['second_class']) &
    (valid_df['prob_difference'] < 0.15)
]

In [ ]:
print(anomalies.head(10))

In [ ]:
anomalies[['ram',
           'battery_power',
           'px_height',
           'px_width',
           'real_class',
           'second_class',
           'prob_difference']].head(10)

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    valid_df['ram'],
    valid_df['real_class'],
    alpha=0.3,
    label='Обычные'
)

plt.scatter(
    anomalies['ram'],
    anomalies['real_class'],
    s=100,
    label='Аномалии'
)

plt.xlabel('RAM')

plt.ylabel('Класс цены')

plt.title('Аномальные телефоны')

plt.legend()

plt.grid()

plt.show()

In [ ]:
test_predictions = svm_model.predict(X_test_scaled)
print(test_predictions)

In [ ]:
submission = test.copy()

submission['price_range'] = test_predictions

submission.head()